# 02 — Retrieval

**Class 5 deliverable.** Use at least two retrieval modes, show how we know
retrieval is *complete*, and preserve raw inputs.

This project uses **three** modes:

| Mode | Source | Implementation |
|---|---|---|
| **FILE** (Parquet over HTTPS) | TLC trip extracts | `ingest.fetch_trip_file` |
| **FILE** (CSV) | taxi zone lookup | `ingest.fetch_zone_lookup` |
| **API** (REST/JSON) | Open-Meteo hourly weather | `ingest.fetch_weather` |
| **SQL** | everything downstream | DuckDB over the raw files |

In [1]:
import sys, json, duckdb, pandas as pd
sys.path.insert(0, '../src')
from pipeline.config import load_config
cfg = load_config('../config/pipeline.yaml')
con = duckdb.connect()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
RAW, MARTS, STG = '../data/raw', '../data/marts', '../data/staging'

## 1. Raw inputs are preserved and provenance is recorded

Every retrieved file is written byte-for-byte and recorded in a manifest with its
source URL, SHA-256 and size. Nothing downstream ever modifies `data/raw/`.

In [2]:
manifest = json.load(open(f'{RAW}/manifest.json'))
m = pd.DataFrame(manifest).T.reset_index(names='key')
m['MB'] = (m.bytes / 1e6).round(2)
m['sha256'] = m.sha256.str[:16] + '...'
display(m[['key', 'MB', 'sha256', 'retrieved_utc']].head(15))
print(f"{len(m)} raw artefacts recorded, {m.MB.sum():.0f} MB total")

,key,MB,sha256,retrieved_utc
0,trips/green/2024-01,1.36,1512a953bed564ac...,2026-09-25T11:20:36.400189+00:00
1,trips/green/2024-02,1.28,cb7eb405ee7e795c...,2026-09-25T11:21:15.878074+00:00
2,trips/green/2024-03,1.37,43dedd66dd556acf...,2026-09-25T11:21:30.557553+00:00
3,trips/green/2024-04,1.35,af174977258a02f8...,2026-09-25T11:21:44.662436+00:00
4,trips/green/2024-05,1.45,aa8b1965ed36f0bd...,2026-09-25T11:22:01.276691+00:00
5,trips/green/2024-06,1.33,47e7c37dd39069c4...,2026-09-25T11:22:13.920459+00:00
6,trips/yellow/2024-01,49.96,c4d59da7bbc8abae...,2026-09-25T11:20:35.705184+00:00
7,trips/yellow/2024-02,50.35,c76c43c18c6c6664...,2026-09-25T11:21:15.120173+00:00
8,trips/yellow/2024-03,60.08,2d4cdc8fb96726cd...,2026-09-25T11:21:29.821993+00:00
9,trips/yellow/2024-04,59.13,ce37f736c7cf9e92...,2026-09-25T11:21:43.957767+00:00


18 raw artefacts recorded, 350 MB total


The checksum is what makes retrieval *auditable*: anyone can verify that the file
analysed is the file published. It is also what lets the pipeline safely skip a
re-download — the idempotency check compares recorded size against what is on disk.

## 2. Retrieval mode 1 — FILE (Parquet over HTTPS)

In [3]:
# Downloads are atomic (.part -> rename) and idempotent. Calling ingest again
# re-fetches nothing, which is why a rerun is cheap and safe to repeat.
import time
from pipeline.ingest import fetch_trip_file
from pipeline.logging_utils import setup_logging
setup_logging(cfg.logs_dir, 'notebook', verbose=True)

t0 = time.time()
path = fetch_trip_file(cfg, 'yellow', '2024-01')
print(f"\nreturned in {time.time()-t0:.3f}s -> {path.name} ({path.stat().st_size/1e6:.1f} MB)")
print("No bytes transferred: the manifest size matched what is on disk.")

2026-09-25 17:02:11 | INFO    |     yellow 2024-01 already present, skipping download



returned in 0.001s -> 2024-01.parquet (50.0 MB)
No bytes transferred: the manifest size matched what is on disk.


## 3. Retrieval mode 2 — API (REST/JSON)

In [4]:
w = json.load(open(f'{RAW}/weather/2024-01.json'))
wdf = pd.DataFrame(w['hourly'])
print(f"rows: {len(wdf)}  (= 24 x 31 days)")
display(wdf.head(3))
print("\nThe raw API response is stored unchanged, so the record of what the API")
print("returned survives even if the API itself changes later.")

rows: 744  (= 24 x 31 days)


,time,temperature_2m,precipitation,snowfall
0,2024-01-01T00:00,2.9,0.0,0.0
1,2024-01-01T01:00,2.5,0.0,0.0
2,2024-01-01T02:00,3.1,0.0,0.0



The raw API response is stored unchanged, so the record of what the API
returned survives even if the API itself changes later.


## 4. Retrieval mode 3 — SQL

In [5]:
# DuckDB queries the raw files directly - no load step, no intermediate copy.
display(con.sql(f'''
    SELECT date_trunc('day', tpep_pickup_datetime) AS day, count(*) AS trips
    FROM read_parquet('{RAW}/trips/yellow/2024-01.parquet')
    WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-01-08'
    GROUP BY 1 ORDER BY 1
''').df())

,day,trips
0,2024-01-01,81013
1,2024-01-02,75519
2,2024-01-03,82427
3,2024-01-04,102901
4,2024-01-05,103178
5,2024-01-06,97117
6,2024-01-07,67543


## 5. How do we know retrieval is *complete*?

A file landing on disk is not evidence of complete retrieval. Six checks run before
any analysis; any failure aborts the month and publishes nothing.

| Check | Question it answers |
|---|---|
| C1 | Did the file arrive, and is it non-empty? |
| C2 | Does it satisfy the schema contract? |
| C3 | Is it readable, with > 0 rows? |
| C4 | Do its contents match the month it claims? |
| C5 | Is the volume continuous with last month? |
| C6 | Does weather cover every hour? |

In [6]:
# C4 in action: does the January file actually contain January?
tot, inside = con.sql(f'''
    SELECT count(*),
           count(*) FILTER (WHERE tpep_pickup_datetime >= '2024-01-01'
                              AND tpep_pickup_datetime <  '2024-02-01')
    FROM read_parquet('{RAW}/trips/yellow/2024-01.parquet')
''').fetchone()
print(f"rows: {tot:,} | inside January: {inside:,} ({100*inside/tot:.3f}%)")
print(f"stray rows from other months: {tot - inside:,}")
print("\nThreshold is 99%. Passing this is what licenses the per-month rerun model:")
print("a trip from another month would be double-counted when that month runs.")

rows: 2,964,624 | inside January: 2,964,606 (99.999%)
stray rows from other months: 18

Threshold is 99%. Passing this is what licenses the per-month rerun model:
a trip from another month would be double-counted when that month runs.


In [7]:
# C5 in action: volume continuity across all six months.
rows = []
for month in cfg.months:
    for fleet in cfg.fleets:
        n = con.sql(f"SELECT count(*) FROM read_parquet('{RAW}/trips/{fleet}/{month}.parquet')").fetchone()[0]
        rows.append({'month': month, 'fleet': fleet, 'rows': n})
v = pd.DataFrame(rows).pivot(index='month', columns='fleet', values='rows')
v['yellow_mom_pct'] = (v.yellow.pct_change() * 100).round(1)
display(v)
print("Tolerance is 30% month-on-month. A collapse or explosion in volume means a")
print("broken extract, not a change in demand - the run stops rather than publishing it.")

fleet,green,yellow,yellow_mom_pct
month,,,
2024-01,56551,2964624,NaN
2024-02,53577,3007526,1.4
2024-03,57457,3582628,19.1
2024-04,56471,3514289,-1.9
2024-05,61003,3723833,6.0
2024-06,54748,3539193,-5.0


Tolerance is 30% month-on-month. A collapse or explosion in volume means a
broken extract, not a change in demand - the run stops rather than publishing it.


In [8]:
# C6: weather completeness for every month.
for month in cfg.months:
    w = json.load(open(f'{RAW}/weather/{month}.json'))
    got = len(w['hourly']['time'])
    days = pd.Period(month).days_in_month
    print(f"{month}: {got:>4} hours | expected {days*24:>4} | {'OK' if got == days*24 else 'MISMATCH'}")

2024-01:  744 hours | expected  744 | OK
2024-02:  696 hours | expected  696 | OK
2024-03:  744 hours | expected  744 | OK
2024-04:  720 hours | expected  720 | OK
2024-05:  744 hours | expected  744 | OK
2024-06:  720 hours | expected  720 | OK


### Where completeness stops

These checks prove **internal** completeness: we received a whole, well-formed,
correctly-labelled extract. They cannot prove **external** completeness — that TLC
published every trip that occurred. No independent count exists to reconcile
against, so this is recorded as assumption **A6**, backed by the C5 volume check
as the practical safeguard.